# Beyond Audio — Reproducible final experiments, Stage 2: Neural networks

**Reconstruction from the team’s original Colab development notebook (Stage 6, cells 62–71).**

This notebook trains two **feed-forward neural networks** on the same six-genre, artist-disjoint MSD-I subset used in Stage 1:

- **CNN early fusion:** 2,048 audio + 2,048 visual features → 4,096 inputs; hidden layers 256 and 64; dropout 0.30 / 0.20; AdamW learning rate 0.0005, weight decay 0.01.
- **MM early fusion:** 200 audio + 200 visual features → 400 inputs; hidden layers 128 and 32; dropout 0.40 / 0.30; AdamW learning rate 0.0003, weight decay 0.05.

Both use train-only scaling, batch size 128, up to 100 epochs, and validation macro-F1 early stopping (patience 12, improvement threshold 1e-4). Random seed 42 is reset before each model, matching the original development notebook. Numerical results may still vary with PyTorch/device/runtime.

**Experimental integrity:** Train/validation data are used for fitting and selection. **Do not execute Section 7 (test evaluation) until both validation runs have been reviewed.** Original development artifacts remain untouched; this notebook saves to `finetuning_experiments/reproducible_stage2_2026/`. This is a reconstructed workflow, not a claim of independently verified reproduction.

## 1. Connect the original project directory and record the environment

Run in Google Colab. If running elsewhere, point `MSDI_PROJECT_DIR` at the project directory. The stage 2 notebook is self-contained and does **not** require executing Stage 1 first.

In [1]:
from pathlib import Path
import os
import sys
import platform
import random
import copy
import json
import numpy as np
import pandas as pd
import joblib
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

SEED = 42
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
CNN_DIR = PROJECT / "finetuning_experiments" / "multimodal_artist_disjoint"
MM_DIR = PROJECT / "finetuning_experiments" / "oramas_representation_experiment"
ORIGINAL_NN_DIR = PROJECT / "finetuning_experiments" / "neural_network_experiments"
OUTPUT = PROJECT / "finetuning_experiments" / "reproducible_stage2_2026"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project:", PROJECT)
print("CNN input directory:", CNN_DIR)
print("MM input directory:", MM_DIR)
print("Separate output directory:", OUTPUT)
print("Python:", platform.python_version())
print("NumPy:", np.__version__, "pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Project: /content/drive/MyDrive/MSD-I_project
CNN input directory: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/multimodal_artist_disjoint
MM input directory: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/oramas_representation_experiment
Separate output directory: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage2_2026
Python: 3.13.16
NumPy: 2.4.6 pandas: 2.2.3
scikit-learn: 1.6.1 PyTorch: 2.14.1+cu130
Device: cuda
GPU: Tesla T4


## 2. Verify the artist-disjoint training and validation data

The original training code used the CNN and MM embeddings stored in two separate folders. Here we cross-check the **track order, genre labels and artists** before processing any data. **No test features are loaded in this section.**

In [2]:
SIZES = {"train": 8242, "val": 1836, "test": 1792}
DIMS = {"cnn_audio": 2048, "cnn_visual": 2048, "mm_audio": 200, "mm_visual": 200}

labels = {}
raw = {}
for split in ("train", "val"):
    p_cnn = CNN_DIR / f"labels_{split}.csv"
    p_mm = MM_DIR / f"labels_{split}.csv"
    if not p_cnn.is_file() or not p_mm.is_file():
        raise FileNotFoundError(f"Missing label CSV for {split}: {p_cnn}, {p_mm}")
    a, b = pd.read_csv(p_cnn), pd.read_csv(p_mm)
    required = ("msd_track_id", "msd_artist_id", "genre", "split")
    for c in required:
        assert c in a.columns and c in b.columns, f"Missing {c} in {split} metadata"
        assert a[c].astype(str).equals(b[c].astype(str)), f"CNN/MM mismatch: {split}, {c}"
    assert len(a) == SIZES[split]
    assert a.msd_track_id.is_unique
    assert (a["split"] == split).all()
    labels[split] = a.reset_index(drop=True)

    def load_one(path, width):
        if not path.is_file():
            raise FileNotFoundError(str(path))
        x = np.load(path, allow_pickle=False)
        assert x.shape == (SIZES[split], width), f"Unexpected shape {path}: {x.shape}"
        assert np.isfinite(x).all(), f"Invalid numerical values: {path}"
        return x

    ca = load_one(CNN_DIR / f"X_{split}_audio.npy", 2048)
    cv = load_one(CNN_DIR / f"X_{split}_visual.npy", 2048)
    ma = load_one(MM_DIR / f"X_{split}_mm-audio.npy", 200)
    mv = load_one(MM_DIR / f"X_{split}_mm-visual.npy", 200)
    # Original Stage 6 concatenated and cast to float32 BEFORE scaling.
    raw["CNN", split] = np.concatenate([ca, cv], axis=1).astype(np.float32)
    raw["MM", split] = np.concatenate([ma, mv], axis=1).astype(np.float32)

assert set(labels["train"].msd_artist_id).isdisjoint(set(labels["val"].msd_artist_id))
assert set(labels["train"].msd_track_id).isdisjoint(set(labels["val"].msd_track_id))
encoder = LabelEncoder()
y = {"train": encoder.fit_transform(labels["train"].genre.astype(str)).astype(np.int64),
     "val": encoder.transform(labels["val"].genre.astype(str)).astype(np.int64)}
assert list(encoder.classes_) == ["Electronic", "Jazz", "Metal", "Pop", "Rap", "Rock"]
for name in ("CNN", "MM"):
    print(f"{name}: train {raw[name, 'train'].shape}, validation {raw[name, 'val'].shape}")
print("Train / validation artist-disjoint: VERIFIED")
print("CNN / MM labels and row alignment: VERIFIED")
print("Genres:", list(encoder.classes_))

CNN: train (8242, 4096), validation (1836, 4096)
MM: train (8242, 400), validation (1836, 400)
Train / validation artist-disjoint: VERIFIED
CNN / MM labels and row alignment: VERIFIED
Genres: ['Electronic', 'Jazz', 'Metal', 'Pop', 'Rap', 'Rock']


## 3. Prepare train-only scaling and define the original MLP architecture

Run this cell, then **stop and check** that the two sets of shapes and the device match expectations before training. Scaling is fitted on training only. The model code matches the two original architectures.

In [3]:
prepared = {}
scalers = {}
for name in ("CNN", "MM"):
    scaler = StandardScaler()
    xtr = scaler.fit_transform(raw[name, "train"]).astype(np.float32)
    xval = scaler.transform(raw[name, "val"]).astype(np.float32)
    assert np.isfinite(xtr).all() and np.isfinite(xval).all()
    assert xtr.shape[0] == len(y["train"]) and xval.shape[0] == len(y["val"])
    scalers[name] = scaler
    prepared[name] = {"train": xtr, "val": xval}
    print(f"{name} scaled: train {xtr.shape}, validation {xval.shape}")

class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, dropout1, dropout2, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

CONFIGS = {
    "CNN": dict(input_dim=4096, hidden1=256, hidden2=64,
                dropout1=0.30, dropout2=0.20, lr=0.0005, weight_decay=0.01),
    "MM": dict(input_dim=400, hidden1=128, hidden2=32,
               dropout1=0.40, dropout2=0.30, lr=0.0003, weight_decay=0.05),
}
for name, config in CONFIGS.items():
    net = FusionMLP(**{k: v for k, v in config.items() if k not in ("lr", "weight_decay")})
    total = sum(p.numel() for p in net.parameters())
    print(f"{name} MLP: {total:,} trainable parameters; input features {config['input_dim']}")

print("Stage 2 data and model definitions are ready. No training has run yet.")

CNN scaled: train (8242, 4096), validation (1836, 4096)
MM scaled: train (8242, 400), validation (1836, 400)
CNN MLP: 1,065,670 trainable parameters; input features 4096
MM MLP: 55,654 trainable parameters; input features 400
Stage 2 data and model definitions are ready. No training has run yet.


## 4. Train CNN fusion MLP — validation-selected checkpoint

This function mirrors the original Stage 6 fitting loop: shuffled 128-sample minibatches, cross-entropy, AdamW, macro-F1 early stopping, and a deep copy of the best weights. Reinitialize seed 42 immediately before constructing each model; **do not overwrite original checkpoints**. Run the next two cells in order.

In [4]:
def reset_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loaders(name):
    train_ds = TensorDataset(torch.from_numpy(prepared[name]["train"]),
                             torch.from_numpy(y["train"]))
    val_ds = TensorDataset(torch.from_numpy(prepared[name]["val"]),
                           torch.from_numpy(y["val"]))
    tr_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
    va_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)
    return train_ds, val_ds, tr_loader, va_loader


def evaluate_class_predictions(model, matrix, batch_size=256):
    model.eval()
    out = []
    with torch.no_grad():
        for j in range(0, len(matrix), batch_size):
            x = torch.from_numpy(matrix[j:j+batch_size]).to(DEVICE)
            out.append(model(x).argmax(dim=1).cpu().numpy())
    return np.concatenate(out)


def train_fusion_nn(name):
    config = CONFIGS[name]
    tr_ds, va_ds, tr_loader, va_loader = make_loaders(name)
    # The original development notebook resets the global seed here,
    # before model initialization and before the first DataLoader shuffle.
    reset_seed(SEED)
    model = FusionMLP(**{k: v for k, v in config.items() if k not in ("lr", "weight_decay")}).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    best_f1, best_epoch, best_state = -float("inf"), 0, None
    stale = 0
    history = []

    for epoch in range(1, 101):
        model.train()
        train_loss_sum = 0.0
        train_pred, train_true = [], []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            train_loss_sum += loss.item() * len(yb)
            train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy())
            train_true.extend(yb.cpu().numpy())
        train_f1 = f1_score(train_true, train_pred, average="macro", zero_division=0)
        train_loss = train_loss_sum / len(tr_ds)

        model.eval()
        val_loss_sum = 0.0
        val_pred, val_true = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                val_loss_sum += loss.item() * len(yb)
                val_pred.extend(logits.argmax(dim=1).cpu().numpy())
                val_true.extend(yb.cpu().numpy())
        val_f1 = f1_score(val_true, val_pred, average="macro", zero_division=0)
        val_loss = val_loss_sum / len(va_ds)
        val_acc = accuracy_score(val_true, val_pred)
        val_bal_acc = balanced_accuracy_score(val_true, val_pred)
        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                            train_macro_f1=train_f1, val_macro_f1=val_f1,
                            val_accuracy=val_acc, val_balanced_accuracy=val_bal_acc))
        if val_f1 > best_f1 + 1e-4:
            best_f1, best_epoch = val_f1, epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        print(f"{name} epoch {epoch:03d} | Train loss {train_loss:.4f} | Val loss {val_loss:.4f} | Train F1 {train_f1:.4f} | Val F1 {val_f1:.4f}")
        if stale >= 12:
            print(f"{name}: early stopping at epoch {epoch}")
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    assert f1_score(y["val"], evaluate_class_predictions(model, prepared[name]["val"]), average="macro") == best_f1 or abs(f1_score(y["val"], evaluate_class_predictions(model, prepared[name]["val"]), average="macro")-best_f1) < 1e-12

    OUTPUT.mkdir(parents=True, exist_ok=True)
    ckpt = dict(model_state_dict=best_state, best_epoch=best_epoch,
                best_val_macro_f1=best_f1, input_dim=config["input_dim"],
                num_classes=6, hidden1=config["hidden1"], hidden2=config["hidden2"],
                dropout1=config["dropout1"], dropout2=config["dropout2"], seed=SEED)
    torch.save(ckpt, OUTPUT / f"{name.lower()}_fusion_mlp_best.pt")
    joblib.dump(scalers[name], OUTPUT / f"{name.lower()}_fusion_mlp_scaler.joblib")
    pd.DataFrame(history).to_csv(OUTPUT / f"{name.lower()}_fusion_mlp_training_history.csv", index=False)
    pd.DataFrame({"msd_track_id": labels["val"].msd_track_id,
                  "genre": labels["val"].genre,
                  "prediction": encoder.inverse_transform(evaluate_class_predictions(model, prepared[name]["val"]))}).to_csv(
        OUTPUT / f"{name.lower()}_fusion_mlp_validation_predictions.csv", index=False)
    print(f"{name}: best epoch {best_epoch}, validation macro-F1 {best_f1:.6f}")
    print("Saved only to:", OUTPUT)
    return model, pd.DataFrame(history)

In [5]:
cnn_model, cnn_history = train_fusion_nn("CNN")

CNN epoch 001 | Train loss 0.9105 | Val loss 0.9677 | Train F1 0.6850 | Val F1 0.6654
CNN epoch 002 | Train loss 0.6077 | Val loss 1.0134 | Train F1 0.7853 | Val F1 0.6660
CNN epoch 003 | Train loss 0.5248 | Val loss 1.0599 | Train F1 0.8171 | Val F1 0.6731
CNN epoch 004 | Train loss 0.4385 | Val loss 1.1280 | Train F1 0.8490 | Val F1 0.6696
CNN epoch 005 | Train loss 0.3815 | Val loss 1.2219 | Train F1 0.8686 | Val F1 0.6738
CNN epoch 006 | Train loss 0.3176 | Val loss 1.3498 | Train F1 0.8919 | Val F1 0.6599
CNN epoch 007 | Train loss 0.2642 | Val loss 1.4241 | Train F1 0.9100 | Val F1 0.6614
CNN epoch 008 | Train loss 0.2218 | Val loss 1.5753 | Train F1 0.9234 | Val F1 0.6537
CNN epoch 009 | Train loss 0.1811 | Val loss 1.6995 | Train F1 0.9417 | Val F1 0.6551
CNN epoch 010 | Train loss 0.1485 | Val loss 1.8133 | Train F1 0.9509 | Val F1 0.6545
CNN epoch 011 | Train loss 0.1313 | Val loss 1.9524 | Train F1 0.9557 | Val F1 0.6540
CNN epoch 012 | Train loss 0.1125 | Val loss 2.1940 | 

## 5. Train MM fusion MLP — validation-selected checkpoint

Run **after** reviewing the CNN training result. Uses the MM architecture, learning rate, and weight decay from the original notebook, with the same seed reset and early stopping criterion.

In [6]:
mm_model, mm_history = train_fusion_nn("MM")

MM epoch 001 | Train loss 1.3634 | Val loss 1.0860 | Train F1 0.5612 | Val F1 0.6604
MM epoch 002 | Train loss 0.8815 | Val loss 0.9343 | Train F1 0.7156 | Val F1 0.6709
MM epoch 003 | Train loss 0.7903 | Val loss 0.9253 | Train F1 0.7259 | Val F1 0.6665
MM epoch 004 | Train loss 0.7542 | Val loss 0.9216 | Train F1 0.7401 | Val F1 0.6762
MM epoch 005 | Train loss 0.7280 | Val loss 0.9320 | Train F1 0.7424 | Val F1 0.6771
MM epoch 006 | Train loss 0.7276 | Val loss 0.9292 | Train F1 0.7503 | Val F1 0.6794
MM epoch 007 | Train loss 0.7170 | Val loss 0.9338 | Train F1 0.7533 | Val F1 0.6811
MM epoch 008 | Train loss 0.7046 | Val loss 0.9330 | Train F1 0.7526 | Val F1 0.6797
MM epoch 009 | Train loss 0.7071 | Val loss 0.9299 | Train F1 0.7522 | Val F1 0.6804
MM epoch 010 | Train loss 0.6937 | Val loss 0.9339 | Train F1 0.7601 | Val F1 0.6804
MM epoch 011 | Train loss 0.6855 | Val loss 0.9280 | Train F1 0.7583 | Val F1 0.6827
MM epoch 012 | Train loss 0.6868 | Val loss 0.9303 | Train F1 0.7

## 6. Verify validation results and original checkpoints

No test data is used. If the original NN checkpoints exist, compare their validation predictions and their selected best epochs with the reconstructed models. Differences should be documented, not hidden. Saved original PyTorch checkpoints are the team's own trusted artifacts; do not load checkpoints from unknown sources.

In [7]:
# Validation-only verification; run after Sections 4 and 5 complete.
comparison_rows = []
for name, new_model, history in [("CNN", cnn_model, cnn_history), ("MM", mm_model, mm_history)]:
    new_pred = evaluate_class_predictions(new_model, prepared[name]["val"])
    new_f1 = f1_score(y["val"], new_pred, average="macro", zero_division=0)
    row = dict(model=name + " Fusion NN", validation_macro_f1=new_f1,
               validation_accuracy=accuracy_score(y["val"], new_pred),
               validation_balanced_accuracy=balanced_accuracy_score(y["val"], new_pred))
    ckpt_path = ORIGINAL_NN_DIR / f"{name.lower()}_fusion_mlp_best.pt"
    scaler_path = ORIGINAL_NN_DIR / f"{name.lower()}_fusion_mlp_scaler.joblib"
    if ckpt_path.is_file() and scaler_path.is_file():
        original_ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        cfg_orig = {k: original_ckpt[k] for k in ("input_dim", "hidden1", "hidden2", "dropout1", "dropout2", "num_classes")}
        original_model = FusionMLP(**cfg_orig).to(DEVICE)
        original_model.load_state_dict(original_ckpt["model_state_dict"])
        original_scaler = joblib.load(scaler_path)
        old_val_scaled = original_scaler.transform(raw[name, "val"]).astype(np.float32)
        old_pred = evaluate_class_predictions(original_model, old_val_scaled)
        reprod_ckpt = torch.load(OUTPUT / f"{name.lower()}_fusion_mlp_best.pt", map_location="cpu", weights_only=True)
        print(f"{name}: original best epoch={original_ckpt['best_epoch']}, reproduced best epoch={reprod_ckpt['best_epoch']}")
        print(f"{name}: original validation macro-F1={f1_score(y['val'], old_pred, average='macro', zero_division=0):.6f}")
        print(f"{name}: differing original vs new validation predictions={int(np.sum(old_pred != new_pred))}")
        print(f"{name}: largest scaler mean difference={np.abs(original_scaler.mean_ - scalers[name].mean_).max():.8g}")
    else:
        print(f"{name}: original checkpoint or scaler unavailable; skipping historical comparison")
    comparison_rows.append(row)
validation_comparison = pd.DataFrame(comparison_rows)
print("\nReproduced validation metrics:")
print(validation_comparison.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
validation_comparison.to_csv(OUTPUT / "stage2_validation_metrics.csv", index=False)

CNN: original best epoch=5, reproduced best epoch=5
CNN: original validation macro-F1=0.673827
CNN: differing original vs new validation predictions=0
CNN: largest scaler mean difference=0
MM: original best epoch=28, reproduced best epoch=28
MM: original validation macro-F1=0.688492
MM: differing original vs new validation predictions=0
MM: largest scaler mean difference=0

Reproduced validation metrics:
        model  validation_macro_f1  validation_accuracy  validation_balanced_accuracy
CNN Fusion NN             0.673827             0.665033                      0.663600
 MM Fusion NN             0.688492             0.677560                      0.678577


## 7. Final held-out test evaluation — **run only after validation review**

The following cell uses the validation-selected checkpoints *without retuning*. It loads the test data for the first time, verifies CNN/MM alignment, reports accuracy, balanced accuracy and macro-F1, then saves its prediction table in the new Stage 2 directory. **Keep the original paper's results separate** and investigate any differences.

In [8]:
# WARNING: Final held-out test; run only once the validation results have been reviewed.
cnn_test_labels = pd.read_csv(CNN_DIR / "labels_test.csv")
mm_test_labels = pd.read_csv(MM_DIR / "labels_test.csv")
for col in ("msd_track_id", "msd_artist_id", "genre", "split"):
    assert cnn_test_labels[col].astype(str).equals(mm_test_labels[col].astype(str)), col
assert len(cnn_test_labels) == 1792
assert cnn_test_labels.msd_track_id.is_unique
assert (cnn_test_labels.split == "test").all()
assert set(cnn_test_labels.msd_artist_id).isdisjoint(set(labels["train"].msd_artist_id))
assert set(cnn_test_labels.msd_artist_id).isdisjoint(set(labels["val"].msd_artist_id))

def test_matrix(name):
    if name == "CNN":
        a = np.load(CNN_DIR / "X_test_audio.npy", allow_pickle=False)
        v = np.load(CNN_DIR / "X_test_visual.npy", allow_pickle=False)
        assert a.shape == (1792, 2048) and v.shape == (1792, 2048)
    else:
        a = np.load(MM_DIR / "X_test_mm-audio.npy", allow_pickle=False)
        v = np.load(MM_DIR / "X_test_mm-visual.npy", allow_pickle=False)
        assert a.shape == (1792, 200) and v.shape == (1792, 200)
    raw_x = np.concatenate([a, v], axis=1).astype(np.float32)
    assert np.isfinite(raw_x).all()
    return raw_x

true_test = encoder.transform(cnn_test_labels.genre.astype(str)).astype(np.int64)
test_predictions = cnn_test_labels.copy()
rows = []
for name, model in (("CNN", cnn_model), ("MM", mm_model)):
    x_test = scalers[name].transform(test_matrix(name)).astype(np.float32)
    pred = evaluate_class_predictions(model, x_test)
    test_predictions[f"{name.lower()}_nn_prediction"] = encoder.inverse_transform(pred)
    rows.append(dict(model=f"{name} Fusion NN",
                     accuracy=100 * accuracy_score(true_test, pred),
                     balanced_accuracy=100 * balanced_accuracy_score(true_test, pred),
                     macro_f1=100 * f1_score(true_test, pred, average="macro", zero_division=0)))
test_metrics = pd.DataFrame(rows)
print(test_metrics.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
test_metrics.to_csv(OUTPUT / "stage2_test_metrics.csv", index=False)
test_predictions.to_csv(OUTPUT / "stage2_test_predictions.csv", index=False)
print("Saved only to:", OUTPUT)

        model  accuracy  balanced_accuracy  macro_f1
CNN Fusion NN     68.58              67.90     67.97
 MM Fusion NN     69.03              68.31     68.49
Saved only to: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage2_2026


In [9]:

# Compare original and reproduced neural-network predictions.
# Diagnostic only: no training or parameter changes.

for name in ("CNN", "MM"):

    original_checkpoint = (
        ORIGINAL_NN_DIR / f"{name.lower()}_fusion_mlp_best.pt"
    )
    original_scaler_path = (
        ORIGINAL_NN_DIR / f"{name.lower()}_fusion_mlp_scaler.joblib"
    )

    checkpoint = torch.load(
        original_checkpoint,
        map_location="cpu",
        weights_only=True
    )

    config = {
        k: checkpoint[k]
        for k in (
            "input_dim", "hidden1", "hidden2",
            "dropout1", "dropout2", "num_classes"
        )
    }

    original_model = FusionMLP(**config).to(DEVICE)
    original_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    original_scaler = joblib.load(original_scaler_path)

    X_test = original_scaler.transform(
        test_matrix(name)
    ).astype(np.float32)

    original_predictions = encoder.inverse_transform(
        evaluate_class_predictions(original_model, X_test)
    )

    reproduced_predictions = test_predictions[
        f"{name.lower()}_nn_prediction"
    ].to_numpy()

    different = np.flatnonzero(
        original_predictions != reproduced_predictions
    )

    print(f"\n{name} Fusion NN")
    print("Different test predictions:", len(different))

    for i in different:
        print(
            cnn_test_labels.iloc[i]["msd_track_id"],
            "| Original:", original_predictions[i],
            "| Reproduced:", reproduced_predictions[i]
        )



CNN Fusion NN
Different test predictions: 0

MM Fusion NN
Different test predictions: 0


## Documentation notes and next steps

- **Original Stage 6 checkpoint names:** `cnn_fusion_mlp_best.pt`, `mm_fusion_mlp_best.pt`; original scalers in the same `neural_network_experiments` directory.
- **Original report's neural-network test metrics** (for comparison only; do not use for training): CNN Fusion NN accuracy 68.58%, balanced accuracy 67.90%, macro-F1 67.97%; MM Fusion NN accuracy 69.03%, balanced accuracy 68.31%, macro-F1 68.49%.
- Environment/device and floating-point numerical differences can affect training trajectories. Use validation metrics and original checkpoints to diagnose any differences before updating report values.
- Stage 3 (separate): reconstruct visual typicality, paired transitions, artist-cluster bootstrap, and robustness checks. That is not included in this notebook.